# Invocations API: Concurrent Vendor Risk Assessment

This notebook mirrors the `concurrent-vendor-risk-assessment` Python scenario and breaks the workflow into runnable learning sections. It uses the Invocations API shape: the caller owns the JSON job payload and selects the scenario per request.

## 1. Notebook Setup

Add this project's `src` directory to the Python path so the local package imports work from Jupyter.

In [ ]:
from pathlib import Path
import sys

def find_project_root(start):
    current = Path(start).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "src" / "review_bot").exists():
            return candidate
        nested = candidate / "invocations-api-review-bot"
        if (nested / "src" / "review_bot").exists():
            return nested
    raise RuntimeError("Could not find invocations-api-review-bot project root.")

PROJECT_ROOT = find_project_root(Path.cwd())
src_path = str(PROJECT_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

PROJECT_ROOT

## 2. Load The Scenario And Helpers

The scenario module contains the enterprise use case, orchestration pattern, sample task, and agent roster.

In [ ]:
from review_bot.scenarios.concurrent_vendor_risk_assessment import SCENARIO
from review_bot.scenarios import SCENARIOS_BY_ID, get_scenario
from review_bot.notebook_helpers import (
    agent_roster,
    default_ollama_options,
    invocation_reference,
    load_sample_payload,
    pattern_anatomy,
    scenario_summary,
)

scenario = SCENARIO
assert SCENARIOS_BY_ID["concurrent-vendor-risk-assessment"] is scenario
assert get_scenario("concurrent-vendor-risk-assessment") is scenario

scenario_summary(scenario)

## 3. Pattern Anatomy

This explains what the orchestration pattern controls and when it fits enterprise application work.

In [ ]:
pattern_anatomy(scenario)

## 4. Flow Diagram

This runtime Mermaid diagram shows the API boundary, orchestration pattern, decisions, actions, and how the scenario agents interact. The helper returns the Mermaid source and displays a Mermaid image in Jupyter.


In [ ]:
from review_bot.diagram_helpers import display_scenario_flow, scenario_flow_diagram

flow_diagram = display_scenario_flow(scenario)
flow_diagram.mermaid


## 5. Agent Roster

Each scenario uses multiple enterprise-role agents. Read the instructions to see what each specialist contributes.

In [ ]:
agent_roster(scenario)

## 6. Load And Validate The Invocation Payload

Invocations own the JSON request shape. The sample payload selects the scenario and carries application-specific fields.

In [ ]:
from review_bot.models import build_review_prompt, parse_review_request

payload = load_sample_payload(PROJECT_ROOT, scenario)
request = parse_review_request(payload)
assert request.scenario == scenario.id
request

## 7. Inspect The Prompt Built From The Payload

The server converts the custom invocation payload into the prompt sent through the orchestration.

In [ ]:
prompt = build_review_prompt(request)
print(prompt)

## 8. Live In-Process Run

This runs the same orchestration without starting the HTTP server. It is the fastest way to study the agent behavior.

In [ ]:
from review_bot.workflows import run_review

ollama_options = default_ollama_options()
response = await run_review(request, **ollama_options)
response.to_dict()

## 9. Read The Structured Result

Unlike Responses, the Invocations API returns an application-owned response object.

In [ ]:
print(f"Scenario: {response.scenario}")
print(f"Pattern: {response.pattern}")
print(f"Agents: {', '.join(response.agents)}")
print("\nSummary:\n")
print(response.summary[:5000])
print("\nRecommendations:\n")
for item in response.recommendations:
    print(f"- {item}")

## 10. Invocations API Boundary

This shows how the same scenario is exposed through the custom `/invocations` endpoint.

In [ ]:
invocation_reference(scenario, request)

## 11. Experiments

Try changing the JSON sample, omitting `scenario` and using `pattern`, or starting the server command from the prior cell and posting the sample JSON from the `samples/` folder.